# UpliftX: EDA and Uplift Modeling
This notebook demonstrates how to load the Hillstrom dataset, perform exploratory data analysis, and train the uplift model.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_data, preprocess_basic
from src.feature_eng import engineer_features
from src.uplift_model import train_uplift_model
from src.evaluator import calculate_qini, get_uplift_by_decile
from src.business_sim import simulate_business_roi

%matplotlib inline

## 1. Data Loading

In [ ]:
df_raw = load_data('../data/raw/hillstrom.csv')
df = preprocess_basic(df_raw)
df.head()

## 2. EDA

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=df, x='treatment_group', hue='visit')
plt.title('Visits by Treatment Group')
plt.show()

## 3. Feature Engineering & Modeling

In [ ]:
X = engineer_features(df, is_training=True, save_path='../models/preprocessor.joblib')
y = df['visit']
t = df['is_treated']

model, results = train_uplift_model(X, y, t, save_dir='../models/')
results.head()

## 4. Evaluation

In [ ]:
qini_df = calculate_qini(results)

plt.figure(figsize=(10,6))
plt.plot(qini_df['n_pop'] / len(qini_df), qini_df['uplift_cumulative'], label='Model Qini')
plt.plot(qini_df['n_pop'] / len(qini_df), qini_df['random_cumulative'], label='Random', linestyle='--')
plt.xlabel('Proportion of Population Targeted')
plt.ylabel('Cumulative Uplift')
plt.title('Qini Curve')
plt.legend()
plt.show()

In [ ]:
decile_df = get_uplift_by_decile(results)
plt.figure(figsize=(10,6))
sns.barplot(data=decile_df, x='decile', y='uplift', palette='viridis')
plt.title('Uplift by Decile')
plt.show()

## 5. Business Simulation

In [ ]:
roi_df = simulate_business_roi(results, treatment_cost=0.5, revenue_per_conversion=50.0)
roi_df